
## Concept

**Algorithm:**
1. The program generates SQL statements.
2. It reads a `.csv` file containing metadata for the SQL statements.
3. For each table listed in the `.csv` (metadataregistry), it creates corresponding SQL statements.
4. It writes a SQL file for each table to `/Workspace/Repos/development/datavault-datamodelling/datascripts/tables/metadataregistry`.

In [0]:
import configparser
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
import json

In [0]:
%run "../helpers/000-log-helper"

In [0]:
# Source
csv_path = "/Volumes/dev/default/quantumcomputing/metadataregistry/"

# Target
sql_target_path = "/Workspace/Repos/development/datavault-datamodelling/datascripts/tables/metadataregistry"

In [0]:
# Get the name of the notebook
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_name = nb_path.split("/")[-1]

# Instantiate the helper class to get the Spark session and logger
helper = HELPER()
spark = helper.get_spark_session(nb_name)
logger = helper.get_logger(nb_name)

In [0]:
# parameters
dbutils.widgets.text("env", "")
dbutils.widgets.text("schema_name", "")
env = dbutils.widgets.get("env")

if env == "":
    env = "dev"
else:
    env = dbutils.widgets.get("env")

schema_name = dbutils.widgets.get("schema_name")
if not schema_name:
    raise logger.error(f"Schema Name Must Be Specified")

catalog_name = env

In [0]:
@dataclass
class CreateMetaSQLs:
    """Create SQL files from metadata registry CSV."""
    csv_path: str
    sql_target_path: str
    spark: object  # SparkSession from HELPER
    logger: object # Logger from HELPER

    def get_csv_files(self) -> List[Path]:
        return [f for f in Path(self.csv_path).glob("*.csv")]

    def read_csv_as_df(self, file_path: Path):
        self.logger.info(f"Reading CSV file: {file_path}")
        return self.spark.read.option("header", True).csv(str(file_path))

    def group_by_table(self, df):
        if "TableName" not in df.columns:
            raise ValueError("Missing 'TableName' column in metadata CSV file.")
        table_names = [row.TableName for row in df.select('TableName').distinct().collect()]
        return [(name, df.filter(df.TableName == name)) for name in table_names]

    def create_table_json(self, table_name: str, group_df) -> Dict:
        description = group_df.select("Description").first()[0]
        collected_rows = group_df.collect()
        pk_candidates = []
        for row in collected_rows:
            key_val_raw = row.Keys if hasattr(row, 'Keys') else None
            key_val_str = str(key_val_raw) if key_val_raw else ""
            key_val_norm = ''.join(key_val_str.strip().upper().split()) # Removes all whitespace
            # print(f"DEBUG: row.Keys (raw)='{key_val_raw}', normalized='{key_val_norm}'")
            if key_val_norm == "PRIMARYKEY":
                pk_candidates.append(row.ColumnName)
        primary_key = pk_candidates[0] if pk_candidates else None
        columns = []
        for row in collected_rows:
            key_val_raw = row.Keys if hasattr(row, 'Keys') else None
            key_val_str = str(key_val_raw) if key_val_raw else ""
            key_val_norm = ''.join(key_val_str.strip().upper().split())
            col = {
                "name": row.ColumnName,
                "type": row.ColumnType,
                "nullable": row.Nullable.lower() == "true" if hasattr(row, 'Nullable') and row.Nullable else False,
            }
            if key_val_norm == "PRIMARYKEY":
                col["constraint"] = "PK"
            if hasattr(row, 'Default') and row.Default:
                col["default"] = row.Default
            col["comment"] = row.Description if hasattr(row, 'Description') and row.Description else ""
            columns.append(col)
        table_json = {
            "table_type": table_name,
            "description": description,
            "is_core": True,
            "primary_key": primary_key,
            "columns": columns
        }
        return table_json

    def create_sql_content(self, table_json: Dict, parameterize: bool = True, parameters: dict = None) -> str:
        pk = table_json.get('primary_key')
        if parameterize:
            table_ref = "${{catalog_name}}.${{env}}_${{schema_name}}.{}".format(table_json['table_type'])
        else:
            cat = parameters['catalog_name']
            envv = parameters.get('env', 'dev')
            sch = parameters['schema_name']
            table_ref = f"{cat}.{envv}_{sch}.{table_json['table_type']}"
        cols = []
        for col in table_json['columns']:
            col_def = f"{col['name']} {col['type']}"
            if col.get('nullable') is False:
                col_def += " NOT NULL"
            if col.get('default'):
                col_def += f" DEFAULT {col['default']}"
            if col['comment']:
                col_def += f" COMMENT '{col['comment']}'"
            cols.append(col_def)
        body = f"-- Predictive optimization: Always optimize this table after large batches\n--  OPTIMIZE {table_ref} ZORDER BY ({pk}) if massive increments\n\nCREATE TABLE IF NOT EXISTS {table_ref} (\n  " + ",\n  ".join(cols)
        if pk:
            body += f",\n  PRIMARY KEY ({pk})"
        body += "\n)\nUSING DELTA"
        if pk:
            body += f"\nCLUSTER BY ({pk}) -- Liquid clustering on PK column for Databricks Runtime 13+"
        body += f"\nCOMMENT '{table_json['description']}'"
        return body

    def save_sql_file(self, table_name: str, sql_content: str):
        out_path = Path(self.sql_target_path) / f"{table_name}.sql"
        with open(out_path, "w") as f:
            f.write(sql_content)
        self.logger.info(f"SQL file saved: {out_path}")

    def process(self):
        csv_files = self.get_csv_files()
        for csv_file in csv_files:
            df = self.read_csv_as_df(csv_file)
            table_groups = self.group_by_table(df)
            for table_name, group_df in table_groups:
                table_json = self.create_table_json(table_name, group_df)
                sql_content = self.create_sql_content(table_json)
                self.save_sql_file(table_name, sql_content)
                self.logger.info(f"Processed table: {table_name}")


In [0]:
# Round 1 Test: Instantiation and CSV file detection
meta_sqls = CreateMetaSQLs(
    csv_path=csv_path,
    sql_target_path=sql_target_path,
    spark=spark,
    logger=logger
)
csv_files = meta_sqls.get_csv_files()
print(f"Round 1: Found CSV files: {csv_files}")
if not csv_files:
    logger.warning("No CSV metadata files found in source!")
else:
    logger.info(f"CSV files detected: {csv_files}")

In [0]:
# Round 2 Test: Group by table and JSON output
if csv_files:
    df = meta_sqls.read_csv_as_df(csv_files[0])
    groups = meta_sqls.group_by_table(df)
    for table_name, group_df in groups:
        table_json = meta_sqls.create_table_json(table_name, group_df)
        print(f"Round 2: Table {table_name} JSON:\n", json.dumps(table_json, indent=2))
else:
    print("Round 2: No CSV files to process.")

In [0]:
# Round 3 Test: SQL File Generation and Verification - Debug PK for MULTIPLE tables
if csv_files:
    for csv_file in csv_files:
        df = meta_sqls.read_csv_as_df(csv_file)
        table_groups = meta_sqls.group_by_table(df)
        for table_name, group_df in table_groups:
            table_json = meta_sqls.create_table_json(table_name, group_df)
            pk_value = table_json.get('primary_key')
            print(f"DEBUG PK for table '{table_name}': {pk_value}")
            test_params = {'catalog_name': catalog_name, 'env': env, 'schema_name': schema_name}
            sql_content = meta_sqls.create_sql_content(table_json, parameterize=False, parameters=test_params)
            print(f"\n----\nGenerated SQL with injected parameters for table '{table_name}':\n{sql_content}\n---\n")
else:
    print("Round 3: No CSV files to process.")

In [0]:
# Entry point for direct Python execution as main
if __name__ == "__main__":
    meta_sqls = CreateMetaSQLs(
        csv_path=csv_path,
        sql_target_path=sql_target_path,
        spark=spark,
        logger=logger
    )
    meta_sqls.process()
